# PHE/VPK NeurIPS Experiment Analysis

Full analysis of experiments A, D, E, G, H, I.  
Run from `experiments/` directory: `jupyter notebook analysis.ipynb`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
from scipy import stats

RESULTS = Path('results')
FIGS    = Path('figures')
FIGS.mkdir(exist_ok=True)

ALGO_ORDER = ['Scrambling','Noise','Combined','Rome','RomeCombined','Ckks','CkksCombined']
ALGO_LABELS = {
    'Scrambling':   'DS',
    'Noise':        'NI',
    'Combined':     'DS+NI',
    'Rome':         'ROME',
    'RomeCombined': 'ROME+NI',
    'Ckks':         'CKKS',
    'CkksCombined': 'CKKS+NI',
}
COLORS = plt.cm.tab10.colors
ALGO_COLOR = {a: COLORS[i] for i, a in enumerate(ALGO_ORDER)}
RIVER_ALPHA = 0.18

plt.rcParams.update({'font.size': 11, 'figure.dpi': 150})

def trial_stats(df, group_cols, metric):
    """Two-level aggregation: query→trial mean, then trial→(mean, std)."""
    trial_means = df.groupby(group_cols + ['trial'])[metric].mean().reset_index()
    return trial_means.groupby(group_cols)[metric].agg(
        **{f'{metric}_mean': 'mean', f'{metric}_std': 'std'}
    ).reset_index()

def river(ax, x, means, stds, color, label, marker='o', lw=2, ms=7, ls='-', zorder=2):
    means = np.asarray(means, dtype=float)
    stds  = np.asarray(stds,  dtype=float)
    ax.plot(x, means, color=color, linewidth=lw, markersize=ms,
            marker=marker, label=label, ls=ls, zorder=zorder)
    ax.fill_between(x, means - stds, means + stds,
                    color=color, alpha=RIVER_ALPHA, linewidth=0, zorder=zorder - 1)

NOISE_ORDER = ['0%','1%','5%','10%','20%','30%','50%','70%']
NOISE_X     = [0, 1, 5, 10, 20, 30, 50, 70]

print('Setup complete.')


## Table 1 — Algorithm Comparison (Exp A, primary paper table)

In [ ]:
df_a = pd.read_csv(RESULTS / 'exp_A_algorithm_comparison.csv')

# Two-level aggregation per algorithm
rows = []
for algo in ALGO_ORDER:
    sub = df_a[df_a['algorithm'] == algo]
    if sub.empty:
        continue
    t = trial_stats(sub, ['algorithm'], 'recall_at_k')
    recall_m = t['recall_at_k_mean'].values[0]
    recall_s = t['recall_at_k_std'].values[0]

    t2 = trial_stats(sub, ['algorithm'], 'ndcg_at_k')
    ndcg_m = t2['ndcg_at_k_mean'].values[0]
    ndcg_s = t2['ndcg_at_k_std'].values[0]

    enc_m = sub['enc_encrypt_us'].mean()
    rows.append({
        'Algorithm':    ALGO_LABELS[algo],
        'Recall@10':    f'{recall_m:.4f} ± {recall_s:.4f}',
        'nDCG@10':      f'{ndcg_m:.4f} ± {ndcg_s:.4f}',
        'Encrypt (μs)': f'{enc_m:.0f}',
        'Score Entropy':f'{sub["score_entropy"].mean():.4f}',
        '_recall_m':    recall_m,
    })

table_a = pd.DataFrame(rows).set_index('Algorithm')
display(table_a.drop(columns=['_recall_m']))
table_a.drop(columns=['_recall_m']).to_csv(FIGS / 'table1_algorithm_comparison.csv')
print('Saved table1_algorithm_comparison.csv')


In [ ]:
# Fig A: Recall@10 and Encrypt latency bar charts
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

algos  = [a for a in ALGO_ORDER if a in df_a['algorithm'].unique()]
labels = [ALGO_LABELS[a] for a in algos]
colors = [ALGO_COLOR[a] for a in algos]

# Left: Recall@10
vals = [trial_stats(df_a[df_a['algorithm']==a], ['algorithm'], 'recall_at_k')['recall_at_k_mean'].values[0] for a in algos]
errs = [trial_stats(df_a[df_a['algorithm']==a], ['algorithm'], 'recall_at_k')['recall_at_k_std'].values[0]  for a in algos]
axes[0].bar(labels, vals, yerr=errs, color=colors, capsize=4)
axes[0].axhline(1.0, ls='--', color='gray', lw=0.8)
axes[0].set_ylabel('Recall@10')
axes[0].set_ylim(0.8, 1.05)
axes[0].tick_params(axis='x', rotation=30)

# Right: Encrypt latency (μs, log scale)
enc_vals = [df_a[df_a['algorithm']==a]['enc_encrypt_us'].mean() for a in algos]
axes[1].bar(labels, enc_vals, color=colors)
axes[1].set_ylabel('Encrypt latency (μs)')
axes[1].set_yscale('log')
axes[1].tick_params(axis='x', rotation=30)

fig.suptitle('Exp A: Algorithm Comparison  (K=10, n=1,000 docs, 3 shards)')
fig.tight_layout()
fig.savefig(FIGS / 'figA_algorithm_comparison.pdf')
plt.show()


## Exp D — Noise–Security Tradeoff

In [ ]:
df_d = pd.read_csv(RESULTS / 'exp_D_noise_tradeoff.csv')
noise_present = [n for n in NOISE_ORDER if n in df_d['noise_label'].unique()]
nx = [NOISE_X[NOISE_ORDER.index(n)] for n in noise_present]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for algo in [a for a in ALGO_ORDER if a in df_d['algorithm'].unique()]:
    sub   = df_d[df_d['algorithm'] == algo]
    color = ALGO_COLOR[algo]
    label = ALGO_LABELS[algo]

    ts_r = trial_stats(sub, ['noise_label'], 'recall_at_k').set_index('noise_label').reindex(noise_present)
    ts_e = trial_stats(sub, ['noise_label'], 'score_entropy').set_index('noise_label').reindex(noise_present)

    river(axes[0], nx, ts_r['recall_at_k_mean'],    ts_r['recall_at_k_std'],    color, label)
    river(axes[1], nx, ts_e['score_entropy_mean'],  ts_e['score_entropy_std'],  color, label)

axes[0].axhline(0.95, ls='--', color='#555', lw=0.8, label='0.95 threshold')
axes[0].set_ylabel('Recall@10  (utility ↑)')
axes[0].set_ylim(0.5, 1.05)
axes[1].set_ylabel('Score Entropy  (security ↑)')

for ax in axes:
    ax.set_xlabel('Noise range (±%)')
    ax.legend(fontsize=8)

fig.suptitle('Exp D: Noise–Security Tradeoff  (fresh DS+NI draw per noise level)')
fig.tight_layout()
fig.savefig(FIGS / 'figD_noise_tradeoff.pdf')
plt.show()


## Exp E — TopK Sensitivity

In [ ]:
df_e = pd.read_csv(RESULTS / 'exp_E_topk_sensitivity.csv')
k_vals = sorted(df_e['top_k'].unique())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for algo in [a for a in ALGO_ORDER if a in df_e['algorithm'].unique()]:
    sub   = df_e[df_e['algorithm'] == algo]
    color = ALGO_COLOR[algo]
    label = ALGO_LABELS[algo]

    ts_r = trial_stats(sub, ['top_k'], 'recall_at_k').set_index('top_k').reindex(k_vals)
    ts_n = trial_stats(sub, ['top_k'], 'ndcg_at_k').set_index('top_k').reindex(k_vals)

    river(axes[0], k_vals, ts_r['recall_at_k_mean'], ts_r['recall_at_k_std'], color, label)
    river(axes[1], k_vals, ts_n['ndcg_at_k_mean'],   ts_n['ndcg_at_k_std'],   color, label)

for ax, title in zip(axes, ['Recall@K', 'nDCG@K']):
    ax.set_xlabel('K')
    ax.set_ylabel(title)
    ax.set_ylim(0.7, 1.05)
    ax.legend(fontsize=8)
    ax.set_xticks(k_vals)

fig.suptitle('Exp E: TopK Sensitivity  (n=1,000 docs, 3 shards)')
fig.tight_layout()
fig.savefig(FIGS / 'figE_topk_sensitivity.pdf')
plt.show()


## Exp G & H — DS-Noise Interaction Analysis

**Hypothesis under test:** Is the high variance in Combined's recall curve (Exp D) caused by DS matrix initialisation quality varying across trials?

**Design:**
- Exp D: each noise level draws a *fresh* (DS, NI) pair — confounds DS-init and noise effects  
- Exp G: DS matrix frozen within each trial; only NI reseeded per noise level  
- Exp H: same as G but explicitly measures DS quality (recall@K at 0% noise) as a per-trial covariate

**Finding:** Freezing DS collapses variance 5–9× at low noise levels.  The DS-init hypothesis is *false* — DS quality = 1.000 ± 0.000 for all 10 Exp H trials.  The true cause of Exp D variance is independent (DS, NI) joint draws that create variable rank-disruptiveness configurations.

In [ ]:
df_g = pd.read_csv(RESULTS / 'exp_G_frozen_ds_noise.csv')
df_h = pd.read_csv(RESULTS / 'exp_H_ds_divergence.csv')

# Common noise levels across all three experiments
noise_common = [n for n in NOISE_ORDER if n in df_d['noise_label'].unique()
                and n in df_g['noise_label'].unique()
                and n in df_h['noise_label'].unique()]
nx_common = [NOISE_X[NOISE_ORDER.index(n)] for n in noise_common]

# -- Panel 1: Exp D vs G vs H mean recall (Combined only) --------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for df, label_tag, ax_title in [
        (df_d, 'Exp D (fresh draw)',  'Exp D — fresh (DS, NI) per level'),
        (df_g, 'Exp G (frozen DS)',   'Exp G — DS frozen, NI reseeded'),
        (df_h, 'Exp H (DS quality covariate)', 'Exp H — frozen DS, 10 trials'),
]:
    sub = df[df['algorithm'] == 'Combined'] if 'algorithm' in df.columns else df
    noise_here = [n for n in noise_common if n in sub['noise_label'].unique()]
    nx_here = [NOISE_X[NOISE_ORDER.index(n)] for n in noise_here]
    ts = trial_stats(sub, ['noise_label'], 'recall_at_k').set_index('noise_label').reindex(noise_here)
    river(axes[0], nx_here, ts['recall_at_k_mean'], ts['recall_at_k_std'],
          ALGO_COLOR['Combined'], label_tag)

axes[0].axhline(0.95, ls='--', color='#555', lw=0.8)
axes[0].set_ylabel('Recall@10')
axes[0].set_ylim(0.5, 1.05)
axes[0].set_xlabel('Noise range (±%)')
axes[0].legend(fontsize=8)
axes[0].set_title('Combined: Exp D vs G vs H')

# -- Panel 2: Std comparison (variance collapse) ------------------------------
std_d = trial_stats(df_d[df_d['algorithm']=='Combined'], ['noise_label'], 'recall_at_k')\
            .set_index('noise_label')['recall_at_k_std'].reindex(noise_common)
std_g = trial_stats(df_g, ['noise_label'], 'recall_at_k')\
            .set_index('noise_label')['recall_at_k_std'].reindex(noise_common)
std_h = trial_stats(df_h, ['noise_label'], 'recall_at_k')\
            .set_index('noise_label')['recall_at_k_std'].reindex(noise_common)

x = np.arange(len(noise_common))
w = 0.25
axes[1].bar(x - w,   std_d.values, w, label='Exp D (fresh)', color='#e07070')
axes[1].bar(x,       std_g.values, w, label='Exp G (frozen DS)', color='#70a0e0')
axes[1].bar(x + w,   std_h.values, w, label='Exp H (frozen DS)', color='#70c070')
axes[1].set_xticks(x); axes[1].set_xticklabels(noise_common, rotation=30)
axes[1].set_ylabel('σ (recall across trials)')
axes[1].set_title('Variance Collapse: Freezing DS')
axes[1].legend(fontsize=8)

# -- Panel 3: Variance reduction factor ---------------------------------------
reduction_g = std_d.values / np.where(std_g.values > 0, std_g.values, np.nan)
reduction_h = std_d.values / np.where(std_h.values > 0, std_h.values, np.nan)
axes[2].plot(nx_common, reduction_g, 'o-', color='#70a0e0', label='D/G ratio')
axes[2].plot(nx_common, reduction_h, 's-', color='#70c070', label='D/H ratio')
axes[2].axhline(1.0, ls='--', color='#888', lw=0.8)
axes[2].set_xlabel('Noise range (±%)')
axes[2].set_ylabel('Variance reduction factor (D / frozen)')
axes[2].set_title('Exp D std / frozen-DS std')
axes[2].legend(fontsize=8)

fig.suptitle('DS-Noise Interaction: Freezing the Rotation Matrix Collapses Variance')
fig.tight_layout()
fig.savefig(FIGS / 'figGH_ds_noise_interaction.pdf')
plt.show()

# Print variance table
var_df = pd.DataFrame({'noise': noise_common,
                       'std_D': std_d.values.round(4),
                       'std_G': std_g.values.round(4),
                       'std_H': std_h.values.round(4),
                       'reduction_G': reduction_g.round(1),
                       'reduction_H': reduction_h.round(1)})
print(var_df.to_string(index=False))


## Exp I — Corpus Size × Noise Tradeoff

In [ ]:
df_i = pd.read_csv(RESULTS / 'exp_I_corpus_size_noise.csv')
corpus_sizes = sorted(df_i['corpus_size'].unique())
cmap = plt.cm.viridis(np.linspace(0.15, 0.85, len(corpus_sizes)))

I_NOISE_ORDER = ['0%','1%','10%','30%','50%','70%']
I_NOISE_X     = [0, 1, 10, 30, 50, 70]

fig, ax = plt.subplots(figsize=(9, 5))
for size, color in zip(corpus_sizes, cmap):
    sub = df_i[df_i['corpus_size'] == size]
    ts  = trial_stats(sub, ['noise_label'], 'recall_at_k').set_index('noise_label').reindex(I_NOISE_ORDER)
    river(ax, I_NOISE_X, ts['recall_at_k_mean'], ts['recall_at_k_std'],
          color, f'n={size:,}', 'o')

ax.axhline(0.95, ls='--', color='#555', lw=0.8, label='0.95 RAG threshold')
ax.set_xlabel('Noise range (±%)')
ax.set_ylabel('Recall@10  (Combined)')
ax.set_ylim(0.82, 1.02)
ax.set_title('Exp I: Corpus Size × Noise Tradeoff\n(shaded = ±1σ across 10 trials)')
ax.legend(fontsize=9, title='Corpus size')
fig.tight_layout()
fig.savefig(FIGS / 'figI_corpus_size_noise.pdf')
plt.show()

# Print the data table
print('\nMean recall by corpus_size × noise_label:')
pivot = df_i.groupby(['corpus_size', 'noise_label'])['recall_at_k'].agg(['mean','std'])
pivot.columns = ['mean', 'std']
pivot = pivot.unstack('noise_label')[['mean','std']].round(4)
print(pivot.to_string())


In [ ]:
print('=' * 60)
print('KEY NUMBERS FOR PAPER')
print('=' * 60)

# Exp A: algorithm comparison
print('\n--- Exp A: Algorithm Comparison (K=10, n=1,000) ---')
for algo in ALGO_ORDER:
    sub = df_a[df_a['algorithm'] == algo]
    if sub.empty: continue
    ts = trial_stats(sub, ['algorithm'], 'recall_at_k')
    m, s = ts['recall_at_k_mean'].values[0], ts['recall_at_k_std'].values[0]
    enc  = sub['enc_encrypt_us'].mean()
    print(f'  {ALGO_LABELS[algo]:<10}  Recall@10={m:.4f}±{s:.4f}  Encrypt={enc:.0f}μs')

# Exp D: recall at 30% noise (operating point)
print('\n--- Exp D: Recall@10 at ±30% noise ---')
for algo in [a for a in ALGO_ORDER if a in df_d['algorithm'].unique()]:
    sub30 = df_d[(df_d['algorithm']==algo) & (df_d['noise_label']=='30%')]
    if sub30.empty: continue
    ts = trial_stats(sub30, ['algorithm'], 'recall_at_k')
    m, s = ts['recall_at_k_mean'].values[0], ts['recall_at_k_std'].values[0]
    print(f'  {ALGO_LABELS[algo]:<12}  {m:.4f}±{s:.4f}')

# Exp G/H: variance collapse
print('\n--- Exp G/H: Variance collapse at ±10% noise (Combined) ---')
if '10%' in noise_common:
    idx = noise_common.index('10%')
    print(f'  Exp D (fresh draw):  σ = {std_d.values[idx]:.4f}')
    print(f'  Exp G (frozen DS):   σ = {std_g.values[idx]:.4f}  ({reduction_g[idx]:.1f}× reduction)')
    print(f'  Exp H (frozen DS):   σ = {std_h.values[idx]:.4f}  ({reduction_h[idx]:.1f}× reduction)')

# Exp I: corpus scaling
print('\n--- Exp I: Recall@10 at ±30% noise across corpus sizes ---')
for size in sorted(df_i['corpus_size'].unique()):
    sub = df_i[(df_i['corpus_size']==size) & (df_i['noise_label']=='30%')]
    ts  = trial_stats(sub, ['corpus_size'], 'recall_at_k')
    m, s = ts['recall_at_k_mean'].values[0], ts['recall_at_k_std'].values[0]
    print(f'  n={size:>5,}  Recall@10={m:.4f}±{s:.4f}')

print('\n' + '=' * 60)


## Paper Summary — Key Numbers

In [ ]:
# Paired Wilcoxon signed-rank test: each algorithm vs DS (Scrambling) baseline
# Unit of comparison: per-trial recall mean (10 trials each)
baseline_algo = 'Scrambling'
baseline_trials = df_a[df_a['algorithm'] == baseline_algo].groupby('trial')['recall_at_k'].mean()

print(f'Statistical tests vs {ALGO_LABELS[baseline_algo]} baseline (Exp A, N=10 trials)\n')
print(f'{"Algorithm":<14} {"Mean recall":>12} {"Δ vs DS":>9} {"W-stat":>8} {"p-value":>10} {"sig":>4}')
print('-' * 65)

test_rows = []
for algo in ALGO_ORDER:
    if algo == baseline_algo:
        continue
    sub = df_a[df_a['algorithm'] == algo]
    if sub.empty:
        continue
    algo_trials = sub.groupby('trial')['recall_at_k'].mean()
    # Align on common trial indices
    common = sorted(set(baseline_trials.index) & set(algo_trials.index))
    b = baseline_trials.loc[common].values
    a = algo_trials.loc[common].values

    mean_r = a.mean()
    delta  = mean_r - b.mean()
    try:
        w, p = stats.wilcoxon(a, b, alternative='two-sided')
    except Exception:
        w, p = np.nan, np.nan

    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f'{ALGO_LABELS[algo]:<14} {mean_r:>12.4f} {delta:>+9.4f} {w:>8.1f} {p:>10.4f} {sig:>4}')
    test_rows.append({'Algorithm': ALGO_LABELS[algo], 'Mean_Recall': mean_r,
                      'Delta_vs_DS': delta, 'p_value': p, 'sig': sig})

pd.DataFrame(test_rows).to_csv(FIGS / 'table_stats_wilcoxon.csv', index=False)
print('\nNote: * p<0.05  ** p<0.01  *** p<0.001  ns = not significant')
print('Saved table_stats_wilcoxon.csv')


## Statistical Tests — Recall@10 vs DS baseline (Exp A)